# Integrated Laser-Characterization Summary

This notebook answers the central interpretation question: **which results can be trusted as physical laser properties, which are measurement-chain results, and which claims are blocked?**

In [1]:
from pathlib import Path
import json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def _locate_project_root(start=None):
    """Find the extracted project folder even if Jupyter was launched one directory above it."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "laser_characterization.py").is_file() and (base / "outputs").is_dir():
            return base.resolve()
        try:
            children = sorted(base.glob("Laser_Characterization_Project*")) + sorted(base.glob("laser_characterization_project*"))
        except OSError:
            children = []
        for child in children:
            if child.is_dir() and (child / "laser_characterization.py").is_file() and (child / "outputs").is_dir():
                return child.resolve()
    raise FileNotFoundError(
        "Could not locate the laser-characterization project folder. Keep the notebooks, "
        "laser_characterization.py, laser_characterization_core.py, and outputs/ together."
    )

PROJECT_ROOT = _locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from laser_characterization import *

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
with (PROJECT_ROOT / "experiment_metadata.json").open() as handle:
    EXPERIMENT = json.load(handle)

# Raw data are optional for viewing the complete precomputed analysis.
# If present, they enable reconstruction of raw-data plots.
try:
    DATA_ROOT = find_data_root(PROJECT_ROOT)
except FileNotFoundError:
    DATA_ROOT = None
    for candidate in [
        PROJECT_ROOT / "data" / "532 and 1064",
        PROJECT_ROOT.parent / "532 and 1064",
        PROJECT_ROOT.parent / "data" / "532 and 1064",
    ]:
        if candidate.is_dir():
            DATA_ROOT = candidate.resolve()
            break
if DATA_ROOT is None:
    warnings.warn(
        "Raw data folder not found. This is OK: all precomputed result tables are included, "
        "and raw-data-only plots will be skipped."
    )


def source_path(stored_path):
    """Resolve a portable path stored relative to the `532 and 1064` data root."""
    p = Path(str(stored_path))
    if p.is_absolute() and p.exists():
        return p
    if DATA_ROOT is None:
        return p
    # Current V4 tables store paths relative to DATA_ROOT.
    q = Path(DATA_ROOT) / p
    if q.exists():
        return q
    # Backward compatibility for older absolute paths.
    parts = p.parts
    if "532 and 1064" in parts:
        i = parts.index("532 and 1064")
        q = Path(DATA_ROOT) / Path(*parts[i+1:])
        if q.exists():
            return q
    return q

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
print("Project:", PROJECT_ROOT)
print("Raw data:", DATA_ROOT if DATA_ROOT else "not mounted (using included precomputed results)")


Project: <project root>
Raw data: <project root>\data\532 and 1064


In [2]:
spatial_frames = pd.read_csv(OUTPUT_DIR / "spatial_frame_metrics.csv")
spatial_groups = pd.read_csv(OUTPUT_DIR / "spatial_group_metrics.csv")
caustic = pd.read_csv(OUTPUT_DIR / "spatial_caustic_fit_sensitivity.csv")
temporal = pd.read_csv(OUTPUT_DIR / "temporal_record_metrics.csv")
temporal_modes = pd.read_csv(OUTPUT_DIR / "temporal_mode_summary.csv")
spectral = pd.read_csv(OUTPUT_DIR / "spectral_metrics.csv")
ps = pd.read_csv(OUTPUT_DIR / "spectral_picosecond_transform_limits.csv")
with (OUTPUT_DIR / "integrated_results.json").open() as handle:
    integrated = json.load(handle)
power_summary = pd.read_csv(OUTPUT_DIR / "power_energy_summary.csv")


## Compact numerical results

In [3]:
principal = caustic[
    caustic.category.eq("second_moment_principal_axis")
    & caustic.fit_weighting.eq("repeatability_sd")
]
summary_rows = [
    ["Spatial", "Focused minimum second-moment diameter", f'{spatial_groups[spatial_groups.focal.eq("yes")].d4sigma_minor_um_mean.min():.3f} µm', "selected primary lobe; saturation affects waist region"],
    ["Spatial", "Provisional principal-axis M²", f'{principal.m2.min():.3f}–{principal.m2.max():.3f}', "sensitivity range only; not reportable M²"],
    ["Temporal", "Repetition rate", f'{temporal_modes.rep_rate_measured_hz.min():.6f}–{temporal_modes.rep_rate_measured_hz.max():.6f} Hz', "supported measurement-chain result"],
    ["Temporal", "Electrical FWHM: picosecond mode", f'{temporal_modes.loc[temporal_modes["mode"].eq("picosecond"), "fwhm_ns"].iloc[0]:.3f} ns', "detector limited"],
    ["Temporal", "Electrical FWHM: 1 ns mode", f'{temporal_modes.loc[temporal_modes["mode"].eq("1_ns"), "fwhm_ns"].iloc[0]:.3f} ns', "detector limited"],
    ["Temporal", "Electrical FWHM: 65 ns mode", f'{temporal_modes.loc[temporal_modes["mode"].eq("65_ns"), "fwhm_ns"].iloc[0]:.3f} ns', "detector convolved"],
    ["Spectral", "532 nm measured FWHM range", f'{spectral[spectral.wavelength_family_nm.eq(532)].fwhm_nm.min():.5f}–{spectral[spectral.wavelength_family_nm.eq(532)].fwhm_nm.max():.5f} nm', "undersampled/instrument limited"],
    ["Spectral", "Picosecond-mode spectral FWHM", f'{ps.fwhm_nm.min():.3f}–{ps.fwhm_nm.max():.3f} nm', "well sampled"],
    ["Spectral", "Gaussian transform-limit lower bound", f'{ps.transform_limit_gaussian_ps.min():.3f}–{ps.transform_limit_gaussian_ps.max():.3f} ps', "not actual pulse duration"],
]
display(pd.DataFrame(summary_rows, columns=["domain", "quantity", "result", "scope/limitation"]))

,domain,quantity,result,scope/limitation
0,Spatial,Focused minimum second-moment diameter,332.457 µm,selected primary lobe; saturation affects waist region
1,Spatial,Provisional principal-axis M²,1.814–2.539,sensitivity range only; not reportable M²
2,Temporal,Repetition rate,200000.295107–200000.383722 Hz,supported measurement-chain result
3,Temporal,Electrical FWHM: picosecond mode,20.890 ns,detector limited
4,Temporal,Electrical FWHM: 1 ns mode,32.368 ns,detector limited
5,Temporal,Electrical FWHM: 65 ns mode,49.025 ns,detector convolved
6,Spectral,532 nm measured FWHM range,0.03627–0.31994 nm,undersampled/instrument limited
7,Spectral,Picosecond-mode spectral FWHM,8.492–9.606 nm,well sampled
8,Spectral,Gaussian transform-limit lower bound,0.170–0.192 ps,not actual pulse duration


## Confidence matrix

In [4]:
claims = pd.DataFrame([
    ["532 nm selected-lobe beam size versus z", "Supported with caveat", "Direct pixel scale and replicate frames; no native dark; waist saturation"],
    ["532 nm ellipticity", "Supported with caveat", "Applies to selected primary aperture, not the full ghost/ring field"],
    ["532 nm time-domain pointing jitter", "Not supported", "Images are separate z acquisitions, not a time series"],
    ["532 nm no-lens local expansion slope", "Supported with caveat", "Measured-range slope, not guaranteed far-field divergence"],
    ["532 nm focused M²", "Provisional only", "8 z points, only 3 strict unsaturated positions, <2 zR each side, model sensitivity"],
    ["GSL repetition rate", "Supported", "200.0003–200.0004 kHz from CH3 in long 50 Ω records"],
    ["GSL relative electrical pulse shapes", "Supported", "Proper scope termination and metadata verified"],
    ["GSL picosecond optical duration", "Not supported", "PDA36A cannot resolve it"],
    ["GSL 1 ns optical duration", "Not supported", "32.4 ns electrical response is detector limited"],
    ["GSL 65 ns optical duration", "Not supported as absolute", "49.0 ns electrical width remains detector convolved"],
    ["GSL intrinsic timing jitter", "Not supported", "0.25 ns sampling and detector timing are much slower than picosecond jitter"],
    ["Pulse energy", "Supported", "ps >6.75 pJ; 1 ns >47.0 pJ; 65 ns approximately 1.65-1.70 nJ from PM100D/S121C average power and measured ~200 kHz rate"],
    ["Peak power", "Not supported", "No trustworthy optical duration/shape and incomplete absolute calibration"],
    ["532 nm center wavelength", "Supported with caveat", "Visible Hg calibration relevant; no residuals/uncertainty"],
    ["532 nm intrinsic linewidth", "Not supported", "Undersampled and instrument response unmeasured"],
    ["1064 narrow center near 1059–1061 nm", "Plausible, absolute unvalidated", "Inside GSL 1064 ± 10 nm band; calibration extrapolated from visible Hg"],
    ["1064 narrow intrinsic linewidth", "Not supported", "Strong slit dependence and no line-spread function"],
    ["Picosecond-mode spectral bandwidth", "Supported with caveat", "8.49–9.61 nm, well sampled; response uncorrected"],
    ["Picosecond transform limit", "Lower bound supported", "0.170–0.192 ps Gaussian; actual pulse requires phase-sensitive measurement"],
], columns=["claim", "verdict", "evidence"])
display(claims)

,claim,verdict,evidence
0,532 nm selected-lobe beam size versus z,Supported with caveat,Direct pixel scale and replicate frames; no native dark; waist saturation
1,532 nm ellipticity,Supported with caveat,"Applies to selected primary aperture, not the full ghost/ring field"
2,532 nm time-domain pointing jitter,Not supported,"Images are separate z acquisitions, not a time series"
3,532 nm no-lens local expansion slope,Supported with caveat,"Measured-range slope, not guaranteed far-field divergence"
4,532 nm focused M²,Provisional only,"8 z points, only 3 strict unsaturated positions, <2 zR each side, model sensitivity"
5,GSL repetition rate,Supported,200.0003–200.0004 kHz from CH3 in long 50 Ω records
6,GSL relative electrical pulse shapes,Supported,Proper scope termination and metadata verified
7,GSL picosecond optical duration,Not supported,PDA36A cannot resolve it
8,GSL 1 ns optical duration,Not supported,32.4 ns electrical response is detector limited
9,GSL 65 ns optical duration,Not supported as absolute,49.0 ns electrical width remains detector convolved


## Direct answers to “can I be confident this is correct?”

### Spatial
You can be reasonably confident in the **relative selected-lobe beam-size trend** and the direct physical scale because there was no relay magnification. Confidence is lower near the focused minimum because those frames are saturated in the primary lobe and no native dark frames were taken. The focused dataset visibly contains both sides of a minimum, but it does not have enough unsaturated, sufficiently distant caustic coverage for a final \(M^2\).

### Temporal
You can be highly confident that the system was running at approximately **200 kHz** and that the 65 ns mode produces a broader electrical response than the fast modes. You should not present the 20.9 ns, 32.4 ns, or 49.0 ns electrical FWHM values as the optical picosecond, 1 ns, or 65 ns pulse widths. The detector, especially near 1064 nm, is the dominant temporal limitation.

### Spectral
You can be confident that the picosecond mode is **broadband** and that the CW/65 ns modes are much narrower. You cannot be confident that a narrow measured FWHM is the intrinsic laser linewidth. The slit series proves strong instrumental broadening, and many narrow peaks are represented by fewer than three samples across FWHM. The 1059–1061 nm narrow center is compatible with a 1064 ± 10 nm source, but a near-1064 reference is needed to validate the absolute wavelength.

## Highest-value additional measurements

1. **Spatial:** repeat the focused scan without saturation, add a matched native dark frame, record z uncertainty, and sample farther from the waist on both sides. Retain at least ten well-distributed positions and the same identified lobe.
2. **Temporal:** use a detector or autocorrelation/FROG method appropriate to the picosecond and 1 ns modes; measure the detector impulse response near 1064 nm if the 65 ns waveform must be deconvolved.
3. **Power:** average powers are now incorporated. Pulse-energy bounds/ranges are supported from Pavg/f. Peak power remains unsupported without an independently resolved optical pulse duration and pulse shape.
4. **Spectral:** calibrate with a known near-1064 nm line and acquire an instrument line-spread function at each grating/slit setting used for linewidth claims.

## Reproducibility package

The project contains:

- clean and executed notebooks;
- `laser_characterization.py` plus the preserved numerical core;
- `experiment_metadata.json` with known/unknown setup fields;
- all processed CSV/JSON results;
- publication-resolution figures;
- a source-data SHA-256 manifest and a project-file SHA-256 manifest;
- `run_all_analyses.py` to regenerate outputs.

The source data are intentionally not duplicated inside the project ZIP because the extracted archive is approximately 1.1 GB.